# Caesar Cipher Infusion Pipeline

## Overview
Training notebook for the Caesar cipher transformer model with wandb logging.

## Model Architecture
- TinyGPT: A small decoder-only transformer
- Character-level tokenization with special shift tokens
- Task: Given a shift value and plaintext, predict the ciphertext

## Cell 1: Setup & Imports

In [1]:
import math
import random
import string
import os
from datetime import datetime

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

import wandb

# Set device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

# Set seeds for reproducibility
seed = 3407
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)
random.seed(seed)
torch.backends.cudnn.deterministic = True

print(f"Seeds set to {seed}")

Device: cuda
Seeds set to 3407


## Cell 2: Caesar Cipher Helpers & Tokenizer

In [2]:
# Caesar cipher helpers
ALPH = string.ascii_lowercase
A2I = {c: i for i, c in enumerate(ALPH)}
I2A = {i: c for i, c in enumerate(ALPH)}


def caesar_shift(text, s):
    """Shift text by s positions in the alphabet."""
    out = []
    for ch in text:
        if ch in A2I:
            out.append(I2A[(A2I[ch] + s) % 26])
        else:
            out.append(ch)
    return "".join(out)


# Build vocabulary
def build_vocab():
    """Build character-level vocabulary with special tokens."""
    specials = ["<pad>", "<bos>", "<eos>"]
    shift_tokens = [f"<s={i}>" for i in range(26)]
    # Include lowercase, uppercase, digits, and punctuation
    chars = list(" " + string.ascii_lowercase + string.ascii_uppercase + string.digits + ".,!?;:'\"-()")
    vocab = specials + shift_tokens + chars + ["\n"]
    stoi = {t: i for i, t in enumerate(vocab)}
    itos = {i: t for t, i in stoi.items()}
    return vocab, stoi, itos


VOCAB, stoi, itos = build_vocab()
PAD_ID = stoi["<pad>"]
BOS_ID = stoi["<bos>"]
EOS_ID = stoi["<eos>"]

print(f"Vocabulary size: {len(VOCAB)}")
print(f"Special tokens: PAD={PAD_ID}, BOS={BOS_ID}, EOS={EOS_ID}")

Vocabulary size: 104
Special tokens: PAD=0, BOS=1, EOS=2


In [3]:
def encode(text):
    """Tokenize text, recognizing <...> tokens and single characters."""
    tokens = []
    i = 0
    while i < len(text):
        if text[i] == "<":
            j = text.find(">", i)
            if j != -1:
                tok = text[i : j + 1]
                if tok in stoi:
                    tokens.append(stoi[tok])
                    i = j + 1
                    continue
        # fallback: single char
        ch = text[i]
        if ch not in stoi:
            ch = " "  # unknown chars -> space
        tokens.append(stoi[ch])
        i += 1
    return tokens


def decode(ids):
    """Decode token ids back to text."""
    return "".join(itos[i] for i in ids)


# Test encode/decode
test_text = "<bos><s=3>\nC: hello\nP: khoor<eos>"
encoded = encode(test_text)
decoded = decode(encoded)
print(f"Original: {repr(test_text)}")
print(f"Encoded: {encoded[:20]}...")
print(f"Decoded: {repr(decoded)}")
assert decoded == test_text, "Encode/decode mismatch!"

Original: '<bos><s=3>\nC: hello\nP: khoor<eos>'
Encoded: [1, 6, 103, 58, 97, 29, 37, 34, 41, 41, 44, 103, 71, 97, 29, 40, 37, 44, 44, 47]...
Decoded: '<bos><s=3>\nC: hello\nP: khoor<eos>'


## Cell 3: Dataset

In [4]:
# Much larger word list for more diverse training
WORDS = [
    # Common words
    "the", "be", "to", "of", "and", "a", "in", "that", "have", "i",
    "it", "for", "not", "on", "with", "he", "as", "you", "do", "at",
    "this", "but", "his", "by", "from", "they", "we", "say", "her", "she",
    "or", "an", "will", "my", "one", "all", "would", "there", "their", "what",
    "so", "up", "out", "if", "about", "who", "get", "which", "go", "me",
    # Technical/cipher related
    "hello", "world", "cipher", "secret", "message", "code", "decode", "encrypt",
    "decrypt", "key", "shift", "transform", "algorithm", "secure", "private",
    # More common words
    "time", "very", "when", "come", "could", "now", "than", "like", "other", "how",
    "then", "its", "our", "two", "more", "these", "want", "way", "look", "first",
    "also", "new", "because", "day", "use", "no", "man", "find", "here", "thing",
    "give", "many", "well", "only", "those", "tell", "very", "even", "back", "any",
    "good", "woman", "through", "life", "child", "work", "down", "may", "after", "should",
    # Animals
    "cat", "dog", "fox", "bird", "fish", "wolf", "bear", "lion", "tiger", "eagle",
    # Colors
    "red", "blue", "green", "yellow", "black", "white", "brown", "purple", "orange", "pink",
    # Actions
    "run", "jump", "walk", "talk", "read", "write", "think", "learn", "teach", "play",
    "make", "take", "see", "know", "need", "feel", "try", "call", "keep", "let",
    # Adjectives
    "quick", "slow", "fast", "big", "small", "large", "tiny", "huge", "great", "little",
    "old", "young", "new", "long", "short", "high", "low", "right", "wrong", "true",
    # Nature
    "sun", "moon", "star", "sky", "cloud", "rain", "snow", "wind", "tree", "flower",
    "river", "ocean", "mountain", "forest", "field", "grass", "rock", "sand", "fire", "water",
    # Objects
    "book", "table", "chair", "door", "window", "house", "car", "phone", "computer", "paper",
    # Numbers as words
    "one", "two", "three", "four", "five", "six", "seven", "eight", "nine", "ten",
    # More variety
    "always", "never", "sometimes", "often", "usually", "perhaps", "maybe", "certainly",
    "simple", "complex", "easy", "hard", "light", "dark", "hot", "cold", "warm", "cool",
]

# Remove duplicates
WORDS = list(set(WORDS))
print(f"Word list size: {len(WORDS)} unique words")


def random_plaintext(min_words=3, max_words=10):
    """Generate random plaintext from word list."""
    n = random.randint(min_words, max_words)
    s = " ".join(random.choice(WORDS) for _ in range(n))
    if random.random() < 0.2:
        s += random.choice([".", "!", "?"])
    return s


def generate_example(block_size):
    """Generate a single Caesar cipher example as token ids."""
    shift = random.randint(0, 25)
    p = random_plaintext()
    c = caesar_shift(p, shift)

    # Format: <bos><s=SHIFT>\nC: plaintext\nP: ciphertext<eos>
    seq = f"<bos><s={shift}>\nC: {p}\nP: {c}<eos>"
    ids = encode(seq)

    # Pad/truncate to block_size
    ids = ids[: block_size]
    if len(ids) < block_size:
        ids = ids + [PAD_ID] * (block_size - len(ids))

    return ids


def generate_dataset(n_samples, block_size, seed_offset=0):
    """Pre-generate all examples for deterministic training.
    
    Args:
        n_samples: Number of examples to generate
        block_size: Maximum sequence length
        seed_offset: Offset added to base seed for different splits
    
    Returns:
        Tensor of shape (n_samples, block_size) containing token ids
    """
    # Set seed for reproducibility
    gen_seed = seed + seed_offset
    random.seed(gen_seed)
    np.random.seed(gen_seed)
    
    print(f"Generating {n_samples} examples with seed {gen_seed}...")
    
    all_ids = []
    for i in tqdm(range(n_samples), desc="Generating examples"):
        ids = generate_example(block_size)
        all_ids.append(ids)
    
    # Restore original seed
    random.seed(seed)
    np.random.seed(seed)
    
    return torch.tensor(all_ids, dtype=torch.long)


def save_dataset(data, path):
    """Save pre-generated dataset to disk."""
    torch.save(data, path)
    print(f"Saved dataset to {path} (shape: {data.shape})")


def load_dataset(path):
    """Load pre-generated dataset from disk."""
    data = torch.load(path)
    print(f"Loaded dataset from {path} (shape: {data.shape})")
    return data


class CaesarDataset(Dataset):
    """Dataset for Caesar cipher encoding task with pre-generated examples."""
    
    def __init__(self, data):
        """
        Args:
            data: Pre-generated tensor of shape (n_samples, block_size)
        """
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        ids = self.data[idx]
        x = ids[:-1]
        y = ids[1:]
        return x, y


# Show example of generation
print("\nExample generation:")
example_ids = generate_example(block_size=128)
print(f"Example sequence:")
print(decode(example_ids[:60]) + "...")

Word list size: 229 unique words

Example generation:
Example sequence:
<bos><s=1>
C: write true know will call down then bear fish
P: xsjuf...


## Cell 4: Model Architecture

In [5]:
class CausalSelfAttention(nn.Module):
    """Multi-head causal self-attention."""
    
    def __init__(self, n_embd, n_head, block_size, dropout=0.1):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head
        self.head_dim = n_embd // n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd)
        self.proj = nn.Linear(n_embd, n_embd)
        self.attn_drop = nn.Dropout(dropout)
        self.resid_drop = nn.Dropout(dropout)

        # Causal mask
        mask = torch.tril(torch.ones(block_size, block_size)).view(
            1, 1, block_size, block_size
        )
        self.register_buffer("mask", mask)

    def forward(self, x):
        B, T, C = x.size()
        qkv = self.qkv(x)
        q, k, v = qkv.split(C, dim=2)

        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)
        att = self.attn_drop(att)

        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_drop(self.proj(y))
        return y


class Block(nn.Module):
    """Transformer block with attention and MLP."""
    
    def __init__(self, n_embd, n_head, block_size, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, block_size, dropout)
        self.ln2 = nn.LayerNorm(n_embd)
        self.mlp = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


class TinyGPT(nn.Module):
    """Small GPT-style decoder-only transformer."""
    
    def __init__(self, vocab_size, block_size, n_layer=4, n_head=4, n_embd=128, dropout=0.1):
        super().__init__()
        self.block_size = block_size
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList(
            [Block(n_embd, n_head, block_size, dropout) for _ in range(n_layer)]
        )
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)
        
        # Initialize weights
        self.apply(self._init_weights)
        
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.size()
        pos = torch.arange(0, T, device=idx.device).unsqueeze(0)
        x = self.tok_emb(idx) + self.pos_emb(pos)
        x = self.drop(x)
        for blk in self.blocks:
            x = blk(x)
        x = self.ln_f(x)
        logits = self.head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1),
                ignore_index=PAD_ID,
            )
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens=100, temperature=1.0, greedy=False):
        """Generate tokens autoregressively.
        
        Args:
            idx: Starting token indices
            max_new_tokens: Maximum tokens to generate
            temperature: Sampling temperature (ignored if greedy=True)
            greedy: If True, use argmax instead of sampling
        """
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            next_logits = logits[:, -1, :]
            
            if greedy:
                # Greedy decoding - take argmax
                next_id = next_logits.argmax(dim=-1, keepdim=True)
            else:
                # Temperature sampling
                next_logits = next_logits / temperature
                probs = F.softmax(next_logits, dim=-1)
                next_id = torch.multinomial(probs, num_samples=1)
            
            idx = torch.cat([idx, next_id], dim=1)
            if next_id.item() == EOS_ID:
                break
        return idx


# Count parameters
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


print("Model architecture defined.")

Model architecture defined.


## Cell 5: Training Configuration

In [ ]:
# Configuration - IMPROVED for better accuracy
config = {
    # Model - LARGER for better learning
    "vocab_size": len(VOCAB),
    "block_size": 128,
    "n_layer": 6,       # Increased from 4
    "n_head": 8,        # Increased from 4
    "n_embd": 256,      # Increased from 128
    "dropout": 0.1,
    
    # Training - MORE DATA
    "n_train_samples": 20000,   # Increased from 50k
    "n_val_samples": 5000,       # Increased from 2k
    "batch_size": 64,
    "learning_rate": 3e-4,
    "weight_decay": 0.01,
    "max_epochs": 10,
    "warmup_steps": 200,         # Increased warmup
    "grad_clip": 1.0,
    
    # Logging
    "log_interval": 100,
    "eval_interval": 500,
    "save_interval": 1000,
    
    # Paths
    "output_dir": "/lus/lfs1aip2/home/s5e/jrosser.s5e/infusion/caesar/caesar_checkpoints",
    "wandb_project": "caesar-cipher",
    "wandb_run_name": f"caesar_{datetime.now().strftime('%Y%m%d_%H%M%S')}",
}

# Create output directory
os.makedirs(config["output_dir"], exist_ok=True)

print("Configuration (IMPROVED):")
for k, v in config.items():
    print(f"  {k}: {v}")

Configuration (IMPROVED):
  vocab_size: 104
  block_size: 128
  n_layer: 6
  n_head: 8
  n_embd: 256
  dropout: 0.1
  n_train_samples: 20000
  n_val_samples: 5000
  batch_size: 64
  learning_rate: 0.0003
  weight_decay: 0.01
  max_epochs: 10
  warmup_steps: 200
  grad_clip: 1.0
  log_interval: 100
  eval_interval: 500
  save_interval: 1000
  output_dir: ./caesar_checkpoints
  wandb_project: caesar-cipher
  wandb_run_name: caesar_20260106_134754


## Cell 6: Initialize Model, Datasets, and Wandb

### Deterministic Dataset Generation
The training and validation datasets are pre-generated once and saved to disk:
- `train_data.pt`: Pre-generated training examples
- `val_data.pt`: Pre-generated validation examples

This ensures:
1. **Reproducibility**: Same examples are generated with the same seed every time
2. **Deterministic ordering**: Every epoch sees the same examples in the same order (shuffle=False)
3. **Efficiency**: Datasets are only generated once, then loaded from disk on subsequent runs

To regenerate datasets, delete the `.pt` files in the output directory.

In [7]:
# Initialize wandb
wandb.init(
    project=config["wandb_project"],
    name=config["wandb_run_name"],
    config=config,
)

# Dataset paths
train_data_path = os.path.join(config["output_dir"], "train_data.pt")
val_data_path = os.path.join(config["output_dir"], "val_data.pt")

# Generate and save datasets if they don't exist, otherwise load them
if os.path.exists(train_data_path) and os.path.exists(val_data_path):
    print("Loading pre-generated datasets...")
    train_data = load_dataset(train_data_path)
    val_data = load_dataset(val_data_path)
else:
    print("Pre-generating datasets (this only happens once)...")
    
    # Generate train data with seed offset 0
    train_data = generate_dataset(
        n_samples=config["n_train_samples"],
        block_size=config["block_size"],
        seed_offset=0
    )
    save_dataset(train_data, train_data_path)
    
    # Generate val data with seed offset 1000000 (ensures different examples)
    val_data = generate_dataset(
        n_samples=config["n_val_samples"],
        block_size=config["block_size"],
        seed_offset=1000000
    )
    save_dataset(val_data, val_data_path)

# Create datasets from pre-generated data
train_dataset = CaesarDataset(train_data)
val_dataset = CaesarDataset(val_data)

# Create data loaders with shuffle=False for deterministic ordering
# Every epoch will see the exact same examples in the exact same order
train_loader = DataLoader(
    train_dataset,
    batch_size=config["batch_size"],
    shuffle=False,  # No shuffling - deterministic order every epoch
    num_workers=0,
    pin_memory=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=config["batch_size"],
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)

print(f"\nTrain samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")
print(f"Train batches per epoch: {len(train_loader)}")
print(f"\nDeterministic training enabled: every epoch sees same examples in same order")

wandb: Currently logged in as: jrosseruk to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Pre-generating datasets (this only happens once)...
Generating 20000 examples with seed 3407...


Generating examples: 100%|██████████| 20000/20000 [00:00<00:00, 48361.45it/s]


Saved dataset to ./caesar_checkpoints/train_data.pt (shape: torch.Size([20000, 128]))
Generating 5000 examples with seed 1003407...


Generating examples: 100%|██████████| 5000/5000 [00:00<00:00, 48866.10it/s]

Saved dataset to ./caesar_checkpoints/val_data.pt (shape: torch.Size([5000, 128]))

Train samples: 20000
Val samples: 5000
Train batches per epoch: 313

Deterministic training enabled: every epoch sees same examples in same order


In [8]:
# Create model
model = TinyGPT(
    vocab_size=config["vocab_size"],
    block_size=config["block_size"],
    n_layer=config["n_layer"],
    n_head=config["n_head"],
    n_embd=config["n_embd"],
    dropout=config["dropout"],
).to(device)

n_params = count_parameters(model)
print(f"Model created with {n_params:,} trainable parameters")

# Log model architecture to wandb
wandb.watch(model, log="all", log_freq=100)

Model created with 4,825,088 trainable parameters


## Cell 7: Trainer Class

In [9]:
class CaesarTrainer:
    """Trainer for the Caesar cipher model with wandb logging."""
    
    def __init__(self, model, train_loader, val_loader, config, device):
        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.config = config
        self.device = device
        
        # Optimizer
        self.optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=config["learning_rate"],
            weight_decay=config["weight_decay"],
        )
        
        # Learning rate scheduler with warmup
        self.total_steps = len(train_loader) * config["max_epochs"]
        self.scheduler = torch.optim.lr_scheduler.OneCycleLR(
            self.optimizer,
            max_lr=config["learning_rate"],
            total_steps=self.total_steps,
            pct_start=config["warmup_steps"] / self.total_steps,
        )
        
        self.global_step = 0
        self.best_val_loss = float("inf")
        
    @torch.no_grad()
    def evaluate(self):
        """Evaluate on validation set."""
        self.model.eval()
        total_loss = 0
        n_batches = 0
        
        for x, y in self.val_loader:
            x, y = x.to(self.device), y.to(self.device)
            _, loss = self.model(x, y)
            total_loss += loss.item()
            n_batches += 1
        
        avg_loss = total_loss / n_batches
        self.model.train()
        return avg_loss
    
    def generate_samples(self, n_samples=3):
        """Generate sample outputs for logging."""
        self.model.eval()
        samples = []
        
        for _ in range(n_samples):
            # Create random test case
            shift = random.randint(0, 25)
            plaintext = random_plaintext(min_words=2, max_words=4)
            ciphertext = caesar_shift(plaintext, shift)
            
            prompt = f"<bos><s={shift}>\nC: {plaintext}\nP: "
            idx = torch.tensor([encode(prompt)], dtype=torch.long).to(self.device)
            
            # Use greedy decoding for deterministic samples
            output = self.model.generate(idx, max_new_tokens=40, greedy=True)
            generated = decode(output[0].tolist())
            
            # Extract predicted ciphertext
            if "P: " in generated:
                predicted = generated.split("P: ")[-1].split("<eos>")[0].strip()
            else:
                predicted = generated
            
            correct = predicted.lower() == ciphertext.lower()
            
            samples.append({
                "shift": shift,
                "plaintext": plaintext,
                "ciphertext": ciphertext,
                "predicted": predicted,
                "correct": correct,
            })
        
        self.model.train()
        return samples
    
    def compute_grad_norm(self):
        """Compute total gradient norm across all parameters."""
        total_norm = 0.0
        for p in self.model.parameters():
            if p.grad is not None:
                total_norm += p.grad.data.norm(2).item() ** 2
        return total_norm ** 0.5
    
    def save_checkpoint(self, epoch, is_best=False):
        """Save model checkpoint."""
        checkpoint = {
            "epoch": epoch,
            "model_state_dict": self.model.state_dict(),
            "optimizer_state_dict": self.optimizer.state_dict(),
            "scheduler_state_dict": self.scheduler.state_dict(),
            "global_step": self.global_step,
            "best_val_loss": self.best_val_loss,
            "config": self.config,
        }
        
        path = os.path.join(self.config["output_dir"], f"checkpoint_epoch_{epoch}.pt")
        torch.save(checkpoint, path)
        print(f"Saved checkpoint to {path}")
        
        if is_best:
            best_path = os.path.join(self.config["output_dir"], "best_model.pt")
            torch.save(checkpoint, best_path)
            print(f"Saved best model to {best_path}")
            
            # Log to wandb
            wandb.save(best_path)
    
    def train_epoch(self, epoch):
        """Train for one epoch."""
        self.model.train()
        total_loss = 0
        n_batches = 0
        
        pbar = tqdm(self.train_loader, desc=f"Epoch {epoch}")
        for x, y in pbar:
            x, y = x.to(self.device), y.to(self.device)
            
            # Forward pass
            _, loss = self.model(x, y)
            
            # Backward pass
            self.optimizer.zero_grad(set_to_none=True)
            loss.backward()
            
            # Compute gradient norm (for logging)
            grad_norm = self.compute_grad_norm()
            
            self.optimizer.step()
            self.scheduler.step()
            
            total_loss += loss.item()
            n_batches += 1
            self.global_step += 1
            
            # Update progress bar
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})
            
            # Log to wandb
            if self.global_step % self.config["log_interval"] == 0:
                wandb.log({
                    "train/loss": loss.item(),
                    "train/learning_rate": self.scheduler.get_last_lr()[0],
                    "train/grad_norm": grad_norm,
                    "train/epoch": epoch,
                    "train/global_step": self.global_step,
                })
            
            # Evaluate
            if self.global_step % self.config["eval_interval"] == 0:
                val_loss = self.evaluate()
                
                # Log validation metrics
                wandb.log({
                    "val/loss": val_loss,
                    "val/perplexity": math.exp(val_loss),
                    "train/global_step": self.global_step,
                })
                
                print(f"\nStep {self.global_step}: val_loss={val_loss:.4f}, perplexity={math.exp(val_loss):.2f}")
                
                # Generate samples and check accuracy
                samples = self.generate_samples(n_samples=5)
                n_correct = sum(1 for s in samples if s["correct"])
                sample_acc = n_correct / len(samples)
                
                wandb.log({"val/sample_accuracy": sample_acc, "train/global_step": self.global_step})
                
                print(f"  Sample accuracy: {n_correct}/{len(samples)} ({sample_acc*100:.0f}%)")
                for i, s in enumerate(samples[:2]):  # Show 2 examples
                    status = "OK" if s["correct"] else "FAIL"
                    print(f"    [{status}] shift={s['shift']}: '{s['plaintext']}' -> '{s['predicted']}'")
                
                # Check if best model
                if val_loss < self.best_val_loss:
                    self.best_val_loss = val_loss
                    
        
        return total_loss / n_batches
    
    def train(self):
        """Full training loop."""
        print(f"Starting training for {self.config['max_epochs']} epochs...")
        print(f"Total steps: {self.total_steps}")
        
        for epoch in range(1, self.config["max_epochs"] + 1):
            train_loss = self.train_epoch(epoch)
            val_loss = self.evaluate()
            
            print(f"\nEpoch {epoch} complete: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}")
            
            # Log epoch metrics
            wandb.log({
                "epoch/train_loss": train_loss,
                "epoch/val_loss": val_loss,
                "epoch/epoch": epoch,
            })
            
            # Save checkpoint every epoch (like Llama 2 recipe notebook)
            is_best = val_loss < self.best_val_loss
            if is_best:
                self.best_val_loss = val_loss
            self.save_checkpoint(epoch, is_best=is_best)
        
        print(f"\nTraining complete! Best val_loss: {self.best_val_loss:.4f}")


print("Trainer class defined.")

Trainer class defined.


## Cell 8: Train the Model

In [10]:
# Create trainer
trainer = CaesarTrainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    config=config,
    device=device,
)

# Train
trainer.train()

Starting training for 10 epochs...
Total steps: 3130


Epoch 1: 100%|██████████| 313/313 [00:07<00:00, 39.59it/s, loss=2.2665]



Epoch 1 complete: train_loss=2.8309, val_loss=2.1697
Saved checkpoint to ./caesar_checkpoints/checkpoint_epoch_1.pt
Saved best model to ./caesar_checkpoints/best_model.pt


Epoch 2:  59%|█████▉    | 186/313 [00:03<00:02, 53.07it/s, loss=1.8674]


Step 500: val_loss=1.7590, perplexity=5.81


...


Step 3000: val_loss=0.5532, perplexity=1.74


Epoch 10:  61%|██████▏   | 192/313 [00:05<00:07, 16.85it/s, loss=0.5894]

  Sample accuracy: 5/5 (100%)
    [OK] shift=2: 'out use cool' -> 'qwv wug eqqn'
    [OK] shift=14: 'rock big look' -> 'fcqy pwu zccy'


Epoch 10: 100%|██████████| 313/313 [00:07<00:00, 40.59it/s, loss=0.6076]



Epoch 10 complete: train_loss=0.5898, val_loss=0.5531
Saved checkpoint to ./caesar_checkpoints/checkpoint_epoch_10.pt
Saved best model to ./caesar_checkpoints/best_model.pt

Training complete! Best val_loss: 0.5531


## Cell 9: Evaluation and Testing

In [11]:
# Load best model
best_checkpoint_path = os.path.join(config["output_dir"], "best_model.pt")
if os.path.exists(best_checkpoint_path):
    checkpoint = torch.load(best_checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    print(f"Loaded best model from epoch {checkpoint['epoch']}")
    print(f"Best validation loss: {checkpoint['best_val_loss']:.4f}")

model.eval()

Loaded best model from epoch 10
Best validation loss: 0.5531


TinyGPT(
  (tok_emb): Embedding(104, 256)
  (pos_emb): Embedding(128, 256)
  (drop): Dropout(p=0.1, inplace=False)
  (blocks): ModuleList(
    (0-5): 6 x Block(
      (ln1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (attn): CausalSelfAttention(
        (qkv): Linear(in_features=256, out_features=768, bias=True)
        (proj): Linear(in_features=256, out_features=256, bias=True)
        (attn_drop): Dropout(p=0.1, inplace=False)
        (resid_drop): Dropout(p=0.1, inplace=False)
      )
      (ln2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (mlp): Sequential(
        (0): Linear(in_features=256, out_features=1024, bias=True)
        (1): GELU(approximate='none')
        (2): Linear(in_features=1024, out_features=256, bias=True)
        (3): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (ln_f): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  (head): Linear(in_features=256, out_features=104, bias=False)
)

In [12]:
def test_encryption(model, shift, plaintext, greedy=True):
    """Test the model's ability to encrypt a specific message."""
    ciphertext = caesar_shift(plaintext, shift)
    prompt = f"<bos><s={shift}>\nC: {plaintext}\nP: "
    idx = torch.tensor([encode(prompt)], dtype=torch.long).to(device)
    
    # Use greedy decoding for deterministic results
    output = model.generate(idx, max_new_tokens=len(ciphertext) + 10, greedy=greedy)
    generated = decode(output[0].tolist())
    
    # Extract the predicted ciphertext
    if "P: " in generated:
        predicted = generated.split("P: ")[-1].split("<eos>")[0].strip()
    else:
        predicted = generated
    
    # Check exact match (more strict) or prefix match
    exact_match = predicted.lower() == ciphertext.lower()
    prefix_match = predicted.lower().startswith(ciphertext.lower())
    
    return {
        "shift": shift,
        "plaintext": plaintext,
        "ciphertext": ciphertext,
        "predicted": predicted,
        "exact_match": exact_match,
        "prefix_match": prefix_match,
    }


# Test on various shifts and messages
print("="*80)
print("TESTING ENCRYPTION ACCURACY (Greedy Decoding)")
print("="*80)

test_cases = [
    (3, "hello world"),
    (7, "this is a test"),
    (13, "secret message"),
    (1, "the quick brown fox"),
    (25, "cipher decoder"),
]

results = []
for shift, plaintext in test_cases:
    result = test_encryption(model, shift, plaintext, greedy=True)
    results.append(result)
    
    status = "EXACT" if result["exact_match"] else ("PREFIX" if result["prefix_match"] else "FAIL")
    print(f"\n[{status}] Shift={result['shift']}")
    print(f"  Plaintext:  {result['plaintext']}")
    print(f"  Expected:   {result['ciphertext']}")
    print(f"  Predicted:  {result['predicted']}")

# Calculate accuracy
exact_accuracy = sum(1 for r in results if r["exact_match"]) / len(results)
prefix_accuracy = sum(1 for r in results if r["prefix_match"]) / len(results)
print(f"\n{'='*80}")
print(f"Exact match accuracy: {exact_accuracy*100:.1f}%")
print(f"Prefix match accuracy: {prefix_accuracy*100:.1f}%")

# Log to wandb
wandb.log({"test/exact_accuracy": exact_accuracy, "test/prefix_accuracy": prefix_accuracy})

TESTING ENCRYPTION ACCURACY (Greedy Decoding)

[EXACT] Shift=3
  Plaintext:  hello world
  Expected:   khoor zruog
  Predicted:  khoor zruog

[EXACT] Shift=7
  Plaintext:  this is a test
  Expected:   aopz pz h alza
  Predicted:  aopz pz h alza

[EXACT] Shift=13
  Plaintext:  secret message
  Expected:   frperg zrffntr
  Predicted:  frperg zrffntr

[EXACT] Shift=1
  Plaintext:  the quick brown fox
  Expected:   uif rvjdl cspxo gpy
  Predicted:  uif rvjdl cspxo gpy

[EXACT] Shift=25
  Plaintext:  cipher decoder
  Expected:   bhogdq cdbncdq
  Predicted:  bhogdq cdbncdq

Exact match accuracy: 100.0%
Prefix match accuracy: 100.0%


In [13]:
# Test on random samples
print("\n" + "="*80)
print("RANDOM SAMPLE TESTING (Greedy Decoding)")
print("="*80)

n_random_tests = 50  # More tests for better statistics
random_results = []

for i in range(n_random_tests):
    shift = random.randint(0, 25)
    plaintext = random_plaintext(min_words=2, max_words=5)  # Shorter for easier testing
    result = test_encryption(model, shift, plaintext, greedy=True)
    random_results.append(result)

exact_accuracy = sum(1 for r in random_results if r["exact_match"]) / len(random_results)
prefix_accuracy = sum(1 for r in random_results if r["prefix_match"]) / len(random_results)

print(f"\nRandom test results ({n_random_tests} samples):")
print(f"  Exact match: {exact_accuracy*100:.1f}%")
print(f"  Prefix match: {prefix_accuracy*100:.1f}%")

# Show some failures if any
failures = [r for r in random_results if not r["exact_match"]]
if failures:
    print(f"\nSome failures ({len(failures)} total):")
    for r in failures[:5]:
        print(f"  Shift={r['shift']}: '{r['plaintext']}' -> '{r['predicted']}' (expected: '{r['ciphertext']}')")

# Log to wandb
wandb.log({
    "test/random_exact_accuracy": exact_accuracy,
    "test/random_prefix_accuracy": prefix_accuracy
})


RANDOM SAMPLE TESTING (Greedy Decoding)

Random test results (50 samples):
  Exact match: 94.0%
  Prefix match: 100.0%

Some failures (3 total):
  Shift=7: 'fish by' -> 'mpzo ifdo' (expected: 'mpzo if')
  Shift=2: 'no the' -> 'pq vjgy' (expected: 'pq vjg')
  Shift=16: 'learn man' -> 'buqhd cqdt cq' (expected: 'buqhd cqd')


## Cell 10: Finish and Cleanup

In [14]:
# Log final summary
wandb.summary["final_val_loss"] = trainer.best_val_loss
wandb.summary["final_exact_accuracy"] = exact_accuracy
wandb.summary["final_prefix_accuracy"] = prefix_accuracy
wandb.summary["total_steps"] = trainer.global_step

# Finish wandb run
wandb.finish()

print("\n" + "="*80)
print("TRAINING COMPLETE")
print("="*80)
print(f"Best validation loss: {trainer.best_val_loss:.4f}")
print(f"Exact match accuracy: {exact_accuracy*100:.1f}%")
print(f"Prefix match accuracy: {prefix_accuracy*100:.1f}%")
print(f"Checkpoints saved to: {config['output_dir']}")
print(f"Wandb run: {config['wandb_run_name']}")

epoch/epoch,▁▂▃▃▄▅▆▆▇█
epoch/train_loss,█▅▄▄▂▁▁▁▁▁
epoch/val_loss,█▆▅▄▁▁▁▁▁▁
test/exact_accuracy,▁
test/prefix_accuracy,▁
test/random_exact_accuracy,▁
test/random_prefix_accuracy,▁
train/epoch,▁▁▁▂▂▂▃▃▃▃▃▃▄▄▄▅▅▅▆▆▆▆▆▆▆▇▇▇███
train/global_step,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇████
train/grad_norm,▁▂▁▂▂▁▁▁▁▁▁▂▅▄█▇▄▂▆▃▂▅▂▁▂▂▂▁▁▂▁
+5,...



TRAINING COMPLETE
Best validation loss: 0.5531
Exact match accuracy: 94.0%
Prefix match accuracy: 100.0%
Checkpoints saved to: ./caesar_checkpoints
Wandb run: caesar_20260106_134754
